<a href="https://colab.research.google.com/github/sahanyafernando/EN3150-A03-edge-cnn/blob/Dhilanka/notebooks/02_model_b_lightweight_optimizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — Model B Lightweight CNN and Optimizers

**EN3150 Assignment 03 — Resource-Constrained CNN for Edge Image Classification**  
**Owner:** Dhilanka  
**Environment:** Google Colab + TensorFlow/Keras

## Goal

Build a depthwise-separable CNN below **100,000 trainable parameters** for the **17-class UCI Jute Pest dataset**.

Compare Adam, SGD, and SGD + Momentum using **validation data only**, then evaluate the validation-selected Model B on the test set.

This notebook uses the exact shared 70% / 15% / 15% split produced by Sahanya's dataset notebook.

In [1]:
%pip install -q scikit-learn

In [2]:
import os
import gc
import time
import json
import random
import shutil
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
SEED = 42
IMG_SIZE = 64
BATCH_SIZE = 64

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("SEED =", SEED)
print("IMG_SIZE =", IMG_SIZE)
print("BATCH_SIZE =", BATCH_SIZE)

SEED = 42
IMG_SIZE = 64
BATCH_SIZE = 64


In [5]:
from google.colab import drive
drive.mount("/content/drive")

MEMBER = "dhilanka"

# ----------------------------------------------------
# Personal storage: dataset cache, checkpoints, plots
# ----------------------------------------------------
PERSONAL_ROOT = Path(
    f"/content/drive/MyDrive/EN3150_A03_PERSONAL/{MEMBER}"
)

DATA_ROOT = PERSONAL_ROOT / "dataset_cache" / "jute_pest"
ARTIFACT_ROOT = PERSONAL_ROOT / "artifacts" / "jute_pest"
PLOT_ROOT = PERSONAL_ROOT / "plots" / "jute_pest"

for p in [
    DATA_ROOT,
    ARTIFACT_ROOT,
    PLOT_ROOT,
]:
    p.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------
# Group-shared folder
# ----------------------------------------------------
# Add EN3150_A03_SHARED as a shortcut inside My Drive first.
SHARED_ROOT = Path(
    "/content/drive/MyDrive/EN3150_A03_SHARED"
)

if not SHARED_ROOT.exists():
    raise FileNotFoundError(
        "EN3150_A03_SHARED was not found in My Drive. "
        "Add the shared folder as a shortcut inside My Drive, "
        "then rerun this cell."
    )

RESULT_ROOT = (
    SHARED_ROOT
    / "shared_results_jute_pest"
)

RESULT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

MANIFEST_PATH = (
    RESULT_ROOT
    / "jute_pest_split_manifest.csv"
)

SUMMARY_PATH = (
    RESULT_ROOT
    / "dataset_summary.json"
)

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        "jute_pest_split_manifest.csv is missing. "
        "Sahanya must run the shared data notebook first."
    )

if not SUMMARY_PATH.exists():
    raise FileNotFoundError(
        "dataset_summary.json is missing. "
        "Sahanya must run the shared data notebook first."
    )

print("Personal root:", PERSONAL_ROOT)
print("Dataset cache:", DATA_ROOT)
print("Artifact root:", ARTIFACT_ROOT)
print("Shared results:", RESULT_ROOT)

Mounted at /content/drive
Personal root: /content/drive/MyDrive/EN3150_A03_PERSONAL/dhilanka
Dataset cache: /content/drive/MyDrive/EN3150_A03_PERSONAL/dhilanka/dataset_cache/jute_pest
Artifact root: /content/drive/MyDrive/EN3150_A03_PERSONAL/dhilanka/artifacts/jute_pest
Shared results: /content/drive/MyDrive/EN3150_A03_SHARED/shared_results_jute_pest


In [6]:
# ============================================================
# UCI JUTE PEST DATASET — USE THE EXACT SHARED GROUP SPLIT
# ============================================================

DATASET_URL = (
    "https://archive.ics.uci.edu/static/public/920/"
    "jute+pest+dataset.zip"
)

ZIP_PATH = DATA_ROOT / "jute_pest_dataset.zip"
EXTRACT_DIR = DATA_ROOT / "extracted"

# ------------------------------------------------------------
# 1. Download once to this member's personal Google Drive
# ------------------------------------------------------------
if not ZIP_PATH.exists():

    print("Downloading UCI Jute Pest dataset...")

    temp_zip = ZIP_PATH.with_suffix(".part")

    if temp_zip.exists():
        temp_zip.unlink()

    urllib.request.urlretrieve(
        DATASET_URL,
        temp_zip
    )

    temp_zip.replace(ZIP_PATH)

    print("Download complete.")

else:

    print(
        "Using cached ZIP:",
        ZIP_PATH
    )


if not zipfile.is_zipfile(ZIP_PATH):
    raise RuntimeError(
        "The cached UCI file is not a valid ZIP."
    )


# ------------------------------------------------------------
# 2. Extract outer UCI ZIP
# ------------------------------------------------------------
EXTRACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

outer_marker = (
    EXTRACT_DIR
    / ".outer_extraction_complete"
)

if not outer_marker.exists():

    print("Extracting outer UCI archive...")

    with zipfile.ZipFile(
        ZIP_PATH,
        "r"
    ) as zf:

        zf.extractall(
            EXTRACT_DIR
        )

    outer_marker.write_text(
        "Outer ZIP extracted",
        encoding="utf-8"
    )

    print("Outer archive extracted.")

else:

    print("Outer archive already extracted.")


# ------------------------------------------------------------
# 3. Extract nested ZIP files recursively
# ------------------------------------------------------------
def extract_nested_zips(root_folder):

    processed = set()

    while True:

        nested_zips = [
            p
            for p in root_folder.rglob("*.zip")
            if str(p.resolve()) not in processed
        ]

        if not nested_zips:
            break

        for nested_zip in nested_zips:

            processed.add(
                str(nested_zip.resolve())
            )

            target_folder = (
                nested_zip.parent
                / f"{nested_zip.stem}_extracted"
            )

            marker = (
                target_folder
                / ".extraction_complete"
            )

            if marker.exists():
                continue

            print(
                "Extracting nested ZIP:",
                nested_zip.name
            )

            target_folder.mkdir(
                parents=True,
                exist_ok=True
            )

            if not zipfile.is_zipfile(
                nested_zip
            ):
                continue

            with zipfile.ZipFile(
                nested_zip,
                "r"
            ) as zf:

                zf.extractall(
                    target_folder
                )

            marker.write_text(
                "Nested ZIP extracted",
                encoding="utf-8"
            )


extract_nested_zips(
    EXTRACT_DIR
)


# ------------------------------------------------------------
# 4. Load shared metadata and exact split manifest
# ------------------------------------------------------------
with open(
    SUMMARY_PATH,
    "r"
) as f:

    dataset_summary = json.load(f)


CLASS_NAMES = dataset_summary["classes"]
NUM_CLASSES = int(
    dataset_summary["num_classes"]
)

assert NUM_CLASSES == 17


manifest = pd.read_csv(
    MANIFEST_PATH
)

required_columns = {
    "relative_path",
    "label",
    "class_name",
    "split",
}

assert required_columns.issubset(
    manifest.columns
)


# ------------------------------------------------------------
# 5. Rebuild this member's local Drive paths
# ------------------------------------------------------------
def build_split(split_name):

    part = (
        manifest[
            manifest["split"]
            == split_name
        ]
        .copy()
        .reset_index(drop=True)
    )

    paths = np.asarray(
        [
            str(
                EXTRACT_DIR
                / rel_path
            )
            for rel_path
            in part[
                "relative_path"
            ].astype(str)
        ],
        dtype=str,
    )

    split_labels = (
        part["label"]
        .to_numpy(
            dtype=np.int32
        )
    )

    missing = [
        p
        for p in paths
        if not Path(p).exists()
    ]

    if missing:

        print(
            "Example missing path:",
            missing[0]
        )

        raise FileNotFoundError(
            f"{len(missing)} manifest images were not found. "
            "Make sure the Jute Pest dataset was extracted correctly."
        )

    return paths, split_labels


train_paths, train_labels = build_split(
    "train"
)

val_paths, val_labels = build_split(
    "validation"
)

test_paths, test_labels = build_split(
    "test"
)


print("Classes:", NUM_CLASSES)
print("Train:", len(train_paths))
print("Validation:", len(val_paths))
print("Test:", len(test_paths))


# ------------------------------------------------------------
# 6. Build raw TensorFlow datasets
# ------------------------------------------------------------
def decode_image(path, label):

    image_bytes = tf.io.read_file(
        path
    )

    image = tf.io.decode_image(
        image_bytes,
        channels=3,
        expand_animations=False,
    )

    image.set_shape(
        [None, None, 3]
    )

    return image, label


raw_train = (
    tf.data.Dataset
    .from_tensor_slices(
        (
            train_paths,
            train_labels
        )
    )
    .map(
        decode_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )
)

raw_val = (
    tf.data.Dataset
    .from_tensor_slices(
        (
            val_paths,
            val_labels
        )
    )
    .map(
        decode_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )
)

raw_test = (
    tf.data.Dataset
    .from_tensor_slices(
        (
            test_paths,
            test_labels
        )
    )
    .map(
        decode_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )
)


def count_examples(ds):

    return int(
        tf.data.experimental
        .cardinality(ds)
        .numpy()
    )


print()
print("TensorFlow cardinality")
print("----------------------")
print("Train:", count_examples(raw_train))
print("Validation:", count_examples(raw_val))
print("Test:", count_examples(raw_test))

print()
print(
    "Shared UCI Jute Pest split loaded successfully."
)

Download complete.
Extracting outer UCI archive...
Outer archive extracted.
Extracting nested ZIP: Jute_Pest_Dataset.zip
Classes: 17
Train: 5064
Validation: 1085
Test: 1086

TensorFlow cardinality
----------------------
Train: 5064
Validation: 1085
Test: 1086

Shared UCI Jute Pest split loaded successfully.


In [7]:
AUTOTUNE = tf.data.AUTOTUNE

def preprocess(
    image,
    label
):

    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE],
        antialias=True,
    )

    image = tf.cast(
        image,
        tf.float32
    )

    return image, label


# Cache first, then reshuffle training samples each epoch.
train_ds = (
    raw_train
    .map(
        preprocess,
        num_parallel_calls=AUTOTUNE
    )
    .cache()
    .shuffle(
        4096,
        seed=SEED,
        reshuffle_each_iteration=True
    )
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    raw_val
    .map(
        preprocess,
        num_parallel_calls=AUTOTUNE
    )
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_ds = (
    raw_test
    .map(
        preprocess,
        num_parallel_calls=AUTOTUNE
    )
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print(
    "Training batches:",
    tf.data.experimental
    .cardinality(train_ds)
    .numpy()
)

print(
    "Validation batches:",
    tf.data.experimental
    .cardinality(val_ds)
    .numpy()
)

print(
    "Test batches:",
    tf.data.experimental
    .cardinality(test_ds)
    .numpy()
)

Training batches: 80
Validation batches: 17
Test batches: 17


## Model B architecture

In [8]:
def build_model_b():
    inp=keras.Input((IMG_SIZE,IMG_SIZE,3),name='image'); x=layers.Rescaling(1/255.)(inp)
    x=layers.SeparableConv2D(32,3,padding='same',activation='relu')(x); x=layers.MaxPooling2D()(x)
    x=layers.SeparableConv2D(64,3,padding='same',activation='relu')(x); x=layers.MaxPooling2D()(x)
    x=layers.SeparableConv2D(128,3,padding='same',activation='relu')(x); x=layers.MaxPooling2D()(x)
    x=layers.GlobalAveragePooling2D()(x); x=layers.Dense(64,activation='relu')(x); out=layers.Dense(NUM_CLASSES,name='logits')(x)
    return keras.Model(inp,out,name='Model_B_Lightweight_CNN')
model_b=build_model_b(); model_b.summary(); print('Parameters:',model_b.count_params()); assert model_b.count_params()<100000

Model: "Model_B_Lightweight_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d                │ (None, 64, 64, 32)     │           155 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_1              │ (None, 32, 32, 64)     │         2,400 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_2              │ (None, 16, 16, 128)    │         8,896 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ logits (Dense)                  │ (None, 17)             │         1,105 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,812 (81.30 KB)

 Trainable params: 20,812 (81.30 KB)

 Non-trainable params: 0 (0.00 B)

Parameters: 20812


In [9]:
def estimate_macs_b(model):
    total=0
    for l in model.layers:
        if isinstance(l,layers.SeparableConv2D):
            h,w,cout=map(int,l.output.shape[1:]); cin=int(l.input.shape[-1]); kh,kw=l.kernel_size; dm=l.depth_multiplier; total+=h*w*cin*kh*kw*dm+h*w*(cin*dm)*cout
        elif isinstance(l,layers.Dense): total+=int(l.input.shape[-1])*int(l.units)
    return int(total)
MODEL_B_MACS=estimate_macs_b(model_b); print('Approx MACs:',f'{MODEL_B_MACS:,}')

Approx MACs: 5,149,760


Depthwise-separable convolution first performs spatial filtering independently per input channel, then uses a 1×1 pointwise convolution to mix channels. This is why it can reduce parameters and MACs compared with a standard convolution.

---
### ✅ Git commit checkpoint
Do **not** commit every cell. Commit after this meaningful milestone.

Suggested commit:
```text
feat: implement sub-100k lightweight model B architecture
```
File:
```text
notebooks/02_model_b_lightweight_optimizer.ipynb
```
Beginner workflow: **Colab → File → Download `.ipynb` → replace local notebook → GitHub Desktop → Commit → Push**.

## Optimizer comparison — 20 epochs each

In [ ]:
LOSS_FN=keras.losses.SparseCategoricalCrossentropy(from_logits=True)
class PersistentEpochTimer(keras.callbacks.Callback):
    def __init__(self,csv_path): super().__init__(); self.csv_path=Path(csv_path); self.csv_path.parent.mkdir(parents=True,exist_ok=True)
    def on_epoch_begin(self,epoch,logs=None): self.start=time.perf_counter()
    def on_epoch_end(self,epoch,logs=None):
        row=pd.DataFrame([{'epoch':int(epoch),'seconds':float(time.perf_counter()-self.start)}])
        row.to_csv(self.csv_path,mode='a',header=not self.csv_path.exists(),index=False)
def read_log(run_name):
    p=ARTIFACT_ROOT/run_name/'training_log.csv'
    if not p.exists(): return pd.DataFrame()
    d=pd.read_csv(p)
    return d.drop_duplicates(subset=['epoch'],keep='last').sort_values('epoch') if 'epoch' in d else d
def average_epoch_time(run_name):
    p=ARTIFACT_ROOT/run_name/'epoch_times.csv'
    if not p.exists(): return np.nan
    d=pd.read_csv(p).drop_duplicates(subset=['epoch'],keep='last')
    return float(d.seconds.mean()) if len(d) else np.nan
def fit_resumable(model,run_name,optimizer,epochs):
    rd=ARTIFACT_ROOT/run_name; rd.mkdir(parents=True,exist_ok=True)
    final=rd/'final.keras'; best=rd/'best.keras'; backup=rd/'backup'; log=rd/'training_log.csv'; timing=rd/'epoch_times.csv'
    if final.exists(): print('Completed run found:',run_name); return keras.models.load_model(final)
    model.compile(optimizer=optimizer,loss=LOSS_FN,metrics=[keras.metrics.SparseCategoricalAccuracy(name='accuracy')])
    callbacks=[keras.callbacks.BackupAndRestore(backup_dir=str(backup),save_freq='epoch',delete_checkpoint=False),keras.callbacks.ModelCheckpoint(str(best),monitor='val_accuracy',mode='max',save_best_only=True,verbose=1),keras.callbacks.CSVLogger(str(log),append=True),PersistentEpochTimer(timing)]
    model.fit(train_ds,validation_data=val_ds,epochs=epochs,callbacks=callbacks,verbose=1)
    model.save(final); return model
def reset_run(run_name):
    import shutil
    p=ARTIFACT_ROOT/run_name
    if p.exists(): shutil.rmtree(p)
def plot_history(run_name,prefix):
    d=read_log(run_name)
    plt.figure(figsize=(8,5)); plt.plot(d.epoch+1,d.loss,label='Train'); plt.plot(d.epoch+1,d.val_loss,label='Validation'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title(prefix+' Loss'); plt.legend(); plt.grid(alpha=.25); plt.tight_layout(); plt.savefig(PLOT_ROOT/(prefix.lower().replace(' ','_')+'_loss.png'),dpi=180); plt.show()
    plt.figure(figsize=(8,5)); plt.plot(d.epoch+1,d.accuracy,label='Train'); plt.plot(d.epoch+1,d.val_accuracy,label='Validation'); plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title(prefix+' Accuracy'); plt.legend(); plt.grid(alpha=.25); plt.tight_layout(); plt.savefig(PLOT_ROOT/(prefix.lower().replace(' ','_')+'_accuracy.png'),dpi=180); plt.show()

In [ ]:
runs = {
    "Adam": (
        "model_b_jute_adam",
        lambda: keras.optimizers.Adam(
            1e-3
        )
    ),

    "SGD": (
        "model_b_jute_sgd",
        lambda: keras.optimizers.SGD(
            1e-2
        )
    ),

    "SGD + Momentum": (
        "model_b_jute_momentum",
        lambda: keras.optimizers.SGD(
            1e-2,
            momentum=0.9
        )
    ),
}

for name, (
    run,
    opt_fn
) in runs.items():

    print(
        "\nTRAINING",
        name
    )

    fit_resumable(
        build_model_b(),
        run,
        opt_fn(),
        20
    )

In [ ]:
plt.figure(figsize=(8,5))
for name,(run,_) in runs.items():
    d=read_log(run); plt.plot(d.epoch+1,d.val_loss,label=name)
plt.xlabel('Epoch'); plt.ylabel('Validation loss'); plt.title('Optimizer Comparison — Validation Loss'); plt.legend(); plt.grid(alpha=.25); plt.tight_layout(); plt.savefig(PLOT_ROOT/'model_b_optimizer_val_loss.png',dpi=180); plt.show()
plt.figure(figsize=(8,5))
for name,(run,_) in runs.items():
    d=read_log(run); plt.plot(d.epoch+1,d.val_accuracy,label=name)
plt.xlabel('Epoch'); plt.ylabel('Validation accuracy'); plt.title('Optimizer Comparison — Validation Accuracy'); plt.legend(); plt.grid(alpha=.25); plt.tight_layout(); plt.savefig(PLOT_ROOT/'model_b_optimizer_val_accuracy.png',dpi=180); plt.show()

In [ ]:
summary = []

for name, (
    run,
    _
) in runs.items():

    d = read_log(run)

    summary.append({
        "optimizer": name,
        "best_val_accuracy": float(
            d.val_accuracy.max()
        ),
        "min_val_loss": float(
            d.val_loss.min()
        ),
        "avg_epoch_time_s": float(
            average_epoch_time(run)
        ),
    })


opt_df = (
    pd.DataFrame(summary)
    .sort_values(
        "best_val_accuracy",
        ascending=False
    )
)

display(opt_df)

opt_df.to_csv(
    RESULT_ROOT
    / "model_b_optimizer_comparison.csv",
    index=False,
)

SELECTED_OPTIMIZER = (
    opt_df.iloc[0].optimizer
)

SELECTED_RUN = (
    runs[
        SELECTED_OPTIMIZER
    ][0]
)

print(
    "Selected from validation:",
    SELECTED_OPTIMIZER
)

Momentum adds a velocity term based on previous updates. Discuss whether your measured curves show smoother/faster convergence than plain SGD; do not assume it always wins.

---
### ✅ Git commit checkpoint
Do **not** commit every cell. Commit after this meaningful milestone.

Suggested commit:
```text
exp: compare Adam SGD and momentum on model B
```
File:
```text
notebooks/02_model_b_lightweight_optimizer.ipynb
```
Beginner workflow: **Colab → File → Download `.ipynb` → replace local notebook → GitHub Desktop → Commit → Push**.

## Evaluate the validation-selected Model B

In [ ]:
def get_true_labels(ds):
    return np.concatenate(
        [
            y.numpy()
            for _, y
            in ds
        ]
    )


Y_TEST = get_true_labels(
    test_ds
)


def model_file_size_mb(path):
    return (
        Path(path).stat().st_size
        / (1024 ** 2)
    )


def benchmark_inference_ms_per_image(
    model,
    ds,
    max_batches=10
):
    batches = []
    n = 0

    for i, (x, _) in enumerate(ds):

        if i >= max_batches:
            break

        batches.append(x)
        n += int(x.shape[0])

    if not batches:
        return np.nan

    _ = model(
        batches[0],
        training=False
    )

    start = time.perf_counter()

    for x in batches:
        _ = model(
            x,
            training=False
        )

    return (
        (time.perf_counter() - start)
        * 1000
        / n
    )


def evaluate_model(
    model,
    name,
    save_name
):

    pred = np.argmax(
        model.predict(
            test_ds,
            verbose=0
        ),
        axis=1
    )

    out = {
        "accuracy": float(
            accuracy_score(
                Y_TEST,
                pred
            )
        ),

        "precision_macro": float(
            precision_score(
                Y_TEST,
                pred,
                average="macro",
                zero_division=0
            )
        ),

        "recall_macro": float(
            recall_score(
                Y_TEST,
                pred,
                average="macro",
                zero_division=0
            )
        ),
    }

    print(out)

    print(
        classification_report(
            Y_TEST,
            pred,
            target_names=CLASS_NAMES,
            digits=4,
            zero_division=0
        )
    )

    cm = confusion_matrix(
        Y_TEST,
        pred
    )

    fig, ax = plt.subplots(
        figsize=(14, 12)
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=CLASS_NAMES
    )

    disp.plot(
        ax=ax,
        xticks_rotation=90,
        colorbar=False
    )

    plt.title(
        name
        + " — Confusion Matrix"
    )

    plt.tight_layout()

    plt.savefig(
        PLOT_ROOT
        / save_name,
        dpi=180
    )

    plt.show()

    return out

In [ ]:
best = keras.models.load_model(
    ARTIFACT_ROOT
    / SELECTED_RUN
    / "best.keras"
)

m = evaluate_model(
    best,
    "Model B — " + SELECTED_OPTIMIZER,
    "model_b_jute_confusion_matrix.png"
)

log = read_log(
    SELECTED_RUN
)

res = {
    "dataset": "UCI Jute Pest",
    "num_classes": int(NUM_CLASSES),
    "model": "Model B — Depthwise Separable CNN",
    "owner": "Dhilanka",
    "optimizer": str(SELECTED_OPTIMIZER),
    "parameters": int(
        best.count_params()
    ),
    "trainable_parameters": int(
        sum(
            np.prod(v.shape)
            for v
            in best.trainable_weights
        )
    ),
    "model_size_mb": float(
        model_file_size_mb(
            ARTIFACT_ROOT
            / SELECTED_RUN
            / "final.keras"
        )
    ),
    "estimated_fp32_weight_kb": float(
        best.count_params()
        * 4
        / 1024
    ),
    "best_val_accuracy": float(
        log.val_accuracy.max()
    ),
    "accuracy": m["accuracy"],
    "precision_macro": m[
        "precision_macro"
    ],
    "recall_macro": m[
        "recall_macro"
    ],
    "avg_epoch_time_s": float(
        average_epoch_time(
            SELECTED_RUN
        )
    ),
    "inference_ms_per_image": float(
        benchmark_inference_ms_per_image(
            best,
            test_ds
        )
    ),
    "approx_macs": int(
        MODEL_B_MACS
    ),
}

with open(
    RESULT_ROOT
    / "model_b.json",
    "w"
) as f:

    json.dump(
        res,
        f,
        indent=2
    )

print(
    json.dumps(
        res,
        indent=2
    )
)

## Interpretation
Explain optimizer behavior, Model B parameter/MAC savings, confusion-matrix errors, and the expected trade-off with Model A.

---
### ✅ Git commit checkpoint
Do **not** commit every cell. Commit after this meaningful milestone.

Suggested commit:
```text
analysis: add model B evaluation and optimizer-selected result export
```
File:
```text
notebooks/02_model_b_lightweight_optimizer.ipynb
```
Beginner workflow: **Colab → File → Download `.ipynb` → replace local notebook → GitHub Desktop → Commit → Push**.